# 第二天实操教程：传统机器学习方法在 MOF 性质预测中的应用

本教程涵盖：
1. 数据准备与特征工程
2. 多种机器学习算法训练（Linear, Ridge, SVM, RF, XGBoost, LightGBM）
3. 超参数优化
4. 模型评估与比较
5. SHAP 可解释性分析
6. 案例：CO₂/CH₄ 吸附预测

## 1. 环境准备

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置绘图风格
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# 导入我们的工具
from ml_models.model_trainer import MOFModelTrainer, train_multiple_models
from ml_models.model_evaluator import ModelEvaluator
from ml_models.interpretability import SHAPAnalyzer, PermutationImportance

print("环境准备完成！")
print(f"可用模型: {list(MOFModelTrainer().available_models.keys())}")

## 2. 数据加载与探索

我们将使用模拟的 MOF 数据集，包含几何特征和 CO₂/CH₄ 吸附性质

In [ ]:
# 加载数据
data_path = '../data/examples/mof_adsorption_dataset.csv'

if Path(data_path).exists():
    df = pd.read_csv(data_path)
    print(f"数据集加载成功！")
else:
    # 如果文件不存在，生成模拟数据
    print("生成模拟数据集...")
    from data_processing.generate_sample_data import generate_mof_adsorption_data
    df = generate_mof_adsorption_data(n_samples=1000, save_path=data_path)

print(f"\n数据集形状: {df.shape}")
print(f"\n前5行数据:")
df.head()

In [ ]:
# 数据统计信息
print("数据集统计信息:")
df.describe()

In [ ]:
# 检查缺失值
print("缺失值统计:")
missing = df.isnull().sum()
print(missing[missing > 0])

if missing.sum() == 0:
    print("\n✓ 无缺失值")
else:
    print(f"\n⚠ 发现 {missing.sum()} 个缺失值")

### 2.1 数据可视化

In [ ]:
# 目标变量分布
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# CO2 吸附量
axes[0].hist(df['co2_uptake'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('CO₂ Uptake (mmol/g)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('CO₂ Adsorption Distribution')

# CH4 吸附量
axes[1].hist(df['ch4_uptake'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('CH₄ Uptake (mmol/g)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('CH₄ Adsorption Distribution')

# 选择性
if 'selectivity' in df.columns:
    axes[2].hist(df['selectivity'], bins=50, edgecolor='black', alpha=0.7, color='green')
    axes[2].set_xlabel('CO₂/CH₄ Selectivity')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Selectivity Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# 特征相关性热力图
# 选择数值特征
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# 计算相关性矩阵
corr = df[numeric_cols].corr()

# 绘制热力图
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

### 2.2 特征与目标的关系

In [ ]:
# 选择几个关键特征，绘制与 CO2 吸附量的关系
key_features = ['surface_area', 'pore_volume', 'pld', 'density']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, feature in enumerate(key_features):
    if feature in df.columns:
        axes[i].scatter(df[feature], df['co2_uptake'], alpha=0.5, s=20)
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('CO₂ Uptake (mmol/g)')
        axes[i].set_title(f'{feature} vs CO₂ Uptake')
        axes[i].grid(True, alpha=0.3)

        # 计算相关系数
        corr_coef = df[[feature, 'co2_uptake']].corr().iloc[0, 1]
        axes[i].text(0.05, 0.95, f'r = {corr_coef:.3f}',
                    transform=axes[i].transAxes,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 3. 数据准备

### 3.1 特征选择

In [ ]:
# 定义特征和目标
# 排除目标变量和标识符
exclude_cols = ['mof_id', 'co2_uptake', 'ch4_uptake', 'selectivity']
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"特征列 ({len(feature_cols)}个):")
print(feature_cols)

# 准备数据
X = df[feature_cols]
y_co2 = df['co2_uptake']

print(f"\nX 形状: {X.shape}")
print(f"y 形状: {y_co2.shape}")

### 3.2 划分训练集和测试集

In [ ]:
# 创建训练器并准备数据
trainer = MOFModelTrainer(random_state=42)

X_train, X_test, y_train, y_test = trainer.prepare_data(
    X, y_co2,
    test_size=0.2,
    scaler_type='standard'  # 标准化
)

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

## 4. 模型训练

### 4.1 基线模型：线性回归

In [ ]:
# 训练线性回归模型
print("训练线性回归模型...")
trainer_linear = MOFModelTrainer(random_state=42)
trainer_linear.scaler = trainer.scaler  # 使用相同的 scaler
trainer_linear.feature_names = feature_cols

results_linear = trainer_linear.train(
    X_train, y_train,
    model_type='linear',
    optimize=False,
    cv=5
)

# 评估
evaluator = ModelEvaluator()
y_pred_linear = trainer_linear.predict(X_test)
metrics_linear = evaluator.calculate_metrics(y_test, y_pred_linear)

evaluator.print_metrics(metrics_linear, title="Linear Regression Performance")

In [ ]:
# 可视化预测结果
evaluator.plot_predictions(y_test, y_pred_linear, title="Linear Regression")

### 4.2 Ridge 回归（L2 正则化）

In [ ]:
# 训练 Ridge 回归
print("训练 Ridge 回归模型（超参数优化）...")
trainer_ridge = MOFModelTrainer(random_state=42)
trainer_ridge.scaler = trainer.scaler
trainer_ridge.feature_names = feature_cols

results_ridge = trainer_ridge.train(
    X_train, y_train,
    model_type='ridge',
    optimize=True,  # 超参数优化
    cv=5
)

# 评估
y_pred_ridge = trainer_ridge.predict(X_test)
metrics_ridge = evaluator.calculate_metrics(y_test, y_pred_ridge)

evaluator.print_metrics(metrics_ridge, title="Ridge Regression Performance")

### 4.3 随机森林

In [ ]:
# 训练随机森林
print("训练随机森林模型（超参数优化）...")
print("注意：这可能需要几分钟...")

trainer_rf = MOFModelTrainer(random_state=42)
trainer_rf.scaler = trainer.scaler
trainer_rf.feature_names = feature_cols

results_rf = trainer_rf.train(
    X_train, y_train,
    model_type='rf',
    optimize=True,
    cv=5,
    verbose=1
)

# 评估
y_pred_rf = trainer_rf.predict(X_test)
metrics_rf = evaluator.calculate_metrics(y_test, y_pred_rf)

evaluator.print_metrics(metrics_rf, title="Random Forest Performance")

In [ ]:
# 特征重要性
importance_rf = trainer_rf.get_feature_importance(top_n=15)
print("\nTop 15 重要特征 (Random Forest):")
print(importance_rf)

# 可视化
evaluator.plot_feature_importance(importance_rf, top_n=15)

### 4.4 XGBoost

In [ ]:
# 训练 XGBoost
print("训练 XGBoost 模型（超参数优化）...")
print("注意：这可能需要几分钟...")

trainer_xgb = MOFModelTrainer(random_state=42)
trainer_xgb.scaler = trainer.scaler
trainer_xgb.feature_names = feature_cols

if 'xgboost' in trainer_xgb.available_models:
    results_xgb = trainer_xgb.train(
        X_train, y_train,
        model_type='xgboost',
        optimize=True,
        cv=5,
        verbose=1
    )

    # 评估
    y_pred_xgb = trainer_xgb.predict(X_test)
    metrics_xgb = evaluator.calculate_metrics(y_test, y_pred_xgb)

    evaluator.print_metrics(metrics_xgb, title="XGBoost Performance")

    # 特征重要性
    importance_xgb = trainer_xgb.get_feature_importance(top_n=15)
    evaluator.plot_feature_importance(importance_xgb, top_n=15)
else:
    print("XGBoost 未安装。请运行: pip install xgboost")

### 4.5 LightGBM

In [ ]:
# 训练 LightGBM
print("训练 LightGBM 模型（超参数优化）...")

trainer_lgb = MOFModelTrainer(random_state=42)
trainer_lgb.scaler = trainer.scaler
trainer_lgb.feature_names = feature_cols

if 'lightgbm' in trainer_lgb.available_models:
    results_lgb = trainer_lgb.train(
        X_train, y_train,
        model_type='lightgbm',
        optimize=True,
        cv=5,
        verbose=1
    )

    # 评估
    y_pred_lgb = trainer_lgb.predict(X_test)
    metrics_lgb = evaluator.calculate_metrics(y_test, y_pred_lgb)

    evaluator.print_metrics(metrics_lgb, title="LightGBM Performance")
else:
    print("LightGBM 未安装。请运行: pip install lightgbm")

## 5. 模型比较

In [ ]:
# 收集所有模型结果
all_results = {
    'Linear': {'y_true': y_test, 'y_pred': y_pred_linear},
    'Ridge': {'y_true': y_test, 'y_pred': y_pred_ridge},
    'RF': {'y_true': y_test, 'y_pred': y_pred_rf},
}

if 'xgboost' in trainer_xgb.available_models:
    all_results['XGBoost'] = {'y_true': y_test, 'y_pred': y_pred_xgb}

if 'lightgbm' in trainer_lgb.available_models:
    all_results['LightGBM'] = {'y_true': y_test, 'y_pred': y_pred_lgb}

# 比较 RMSE
evaluator.compare_models(all_results, metric='rmse')

In [ ]:
# 比较 R²
evaluator.compare_models(all_results, metric='r2')

In [ ]:
# 创建性能对比表
comparison_data = []

for model_name, data in all_results.items():
    metrics = evaluator.calculate_metrics(data['y_true'], data['y_pred'])
    comparison_data.append({
        'Model': model_name,
        'R²': metrics['r2'],
        'RMSE': metrics['rmse'],
        'MAE': metrics['mae']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('RMSE')

print("\n模型性能对比:")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)

## 6. 学习曲线分析

诊断过拟合/欠拟合

In [ ]:
# 选择最佳模型（假设是 XGBoost 或 LightGBM）
if 'xgboost' in trainer_xgb.available_models:
    best_trainer = trainer_xgb
    best_name = 'XGBoost'
elif 'lightgbm' in trainer_lgb.available_models:
    best_trainer = trainer_lgb
    best_name = 'LightGBM'
else:
    best_trainer = trainer_rf
    best_name = 'Random Forest'

print(f"绘制 {best_name} 学习曲线...")

# 合并训练集和测试集用于学习曲线
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

evaluator.plot_learning_curve(
    best_trainer.model,
    X_all,
    y_all,
    cv=5
)

## 7. SHAP 可解释性分析

In [ ]:
# 安装 SHAP（如果需要）
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    print("SHAP 未安装。请运行: pip install shap")
    SHAP_AVAILABLE = False

In [ ]:
if SHAP_AVAILABLE:
    # 创建 SHAP 分析器
    print("创建 SHAP 分析器...")
    shap_analyzer = SHAPAnalyzer(
        model=best_trainer.model,
        X=X_test,
        feature_names=feature_cols
    )

    # 计算 SHAP 值
    shap_values = shap_analyzer.compute_shap_values(
        X_explain=X_test,
        max_samples=500  # 限制样本数以加快计算
    )

    print(f"SHAP 值形状: {shap_values.shape}")

### 7.1 SHAP Summary Plot（蜂群图）

In [ ]:
if SHAP_AVAILABLE:
    # 蜂群图：显示所有特征的重要性和影响方向
    shap_analyzer.summary_plot(
        plot_type='dot',
        max_display=20
    )

### 7.2 SHAP Bar Plot（特征重要性）

In [ ]:
if SHAP_AVAILABLE:
    # 条形图：特征重要性排序
    shap_analyzer.summary_plot(
        plot_type='bar',
        max_display=20
    )

### 7.3 SHAP Dependence Plot（特征依赖图）

In [ ]:
if SHAP_AVAILABLE:
    # 绘制最重要特征的依赖图
    importance_df = shap_analyzer.feature_importance(top_n=5)
    top_features = importance_df['feature'].tolist()[:3]

    print(f"绘制前3个重要特征的依赖图: {top_features}")

    for feature in top_features:
        shap_analyzer.dependence_plot(feature)

### 7.4 SHAP Waterfall Plot（单样本解释）

In [ ]:
if SHAP_AVAILABLE:
    # 解释几个样本
    # 选择预测最高、最低和中间的样本
    y_pred_test = best_trainer.predict(X_test)
    
    idx_high = np.argmax(y_pred_test)
    idx_low = np.argmin(y_pred_test)
    idx_med = np.argsort(y_pred_test)[len(y_pred_test)//2]

    print(f"\n样本解释:")
    print(f"最高吸附量样本 (idx={idx_high}): 预测={y_pred_test[idx_high]:.3f}, 真实={y_test.iloc[idx_high]:.3f}")
    print(f"最低吸附量样本 (idx={idx_low}): 预测={y_pred_test[idx_low]:.3f}, 真实={y_test.iloc[idx_low]:.3f}")
    print(f"中等吸附量样本 (idx={idx_med}): 预测={y_pred_test[idx_med]:.3f}, 真实={y_test.iloc[idx_med]:.3f}")

    # 绘制瀑布图
    for idx, name in [(idx_high, '最高'), (idx_low, '最低'), (idx_med, '中等')]:
        print(f"\n{name}吸附量样本的SHAP解释:")
        shap_analyzer.waterfall_plot(sample_idx=idx)

### 7.5 SHAP Feature Importance vs Tree-based Importance

In [ ]:
if SHAP_AVAILABLE:
    # 比较两种特征重要性方法
    shap_importance = shap_analyzer.feature_importance(top_n=15)
    tree_importance = best_trainer.get_feature_importance(top_n=15)

    # 合并
    comparison = pd.merge(
        shap_importance[['feature', 'importance']].rename(columns={'importance': 'SHAP'}),
        tree_importance[['feature', 'importance']].rename(columns={'importance': 'Tree'}),
        on='feature',
        how='outer'
    ).fillna(0)

    # 归一化
    comparison['SHAP'] = comparison['SHAP'] / comparison['SHAP'].max()
    comparison['Tree'] = comparison['Tree'] / comparison['Tree'].max()

    # 绘图
    comparison_plot = comparison.set_index('feature').head(15)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    comparison_plot.plot(kind='barh', ax=ax, width=0.8)
    ax.set_xlabel('Normalized Importance')
    ax.set_ylabel('Feature')
    ax.set_title('SHAP vs Tree-based Feature Importance')
    ax.legend(['SHAP', 'Tree-based'])
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    print("\n特征重要性对比 (归一化):")
    print(comparison.head(15).to_string(index=False))

## 8. 模型保存

In [ ]:
# 保存最佳模型
model_save_path = '../data/examples/best_model_co2.pkl'
best_trainer.save_model(model_save_path)

print(f"\n模型已保存到: {model_save_path}")

## 9. 预测新样本

In [ ]:
# 从测试集中随机选择几个样本进行预测展示
n_samples = 10
sample_indices = np.random.choice(len(X_test), n_samples, replace=False)

X_samples = X_test[sample_indices]
y_true_samples = y_test.iloc[sample_indices]
y_pred_samples = best_trainer.predict(X_samples)

# 创建结果 DataFrame
results_df = pd.DataFrame({
    'True CO₂ Uptake': y_true_samples.values,
    'Predicted CO₂ Uptake': y_pred_samples,
    'Error': y_pred_samples - y_true_samples.values,
    'Relative Error (%)': np.abs(y_pred_samples - y_true_samples.values) / y_true_samples.values * 100
})

print("\n预测示例:")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)
print(f"\n平均绝对误差: {results_df['Error'].abs().mean():.4f}")
print(f"平均相对误差: {results_df['Relative Error (%)'].mean():.2f}%")

## 10. 练习题

### 练习 1：预测 CH₄ 吸附量
重复上述流程，但目标变量改为 `ch4_uptake`

### 练习 2：预测 CO₂/CH₄ 选择性
使用 `selectivity` 作为目标变量，注意可能需要对数变换

### 练习 3：特征工程
尝试创建新的特征：
- `surface_area / density`
- `pore_volume / density`
- `lcd / pld` (孔径比)

观察这些新特征是否能提高模型性能

### 练习 4：超参数调优
尝试手动调整超参数，看能否超越GridSearchCV的结果

### 练习 5：集成学习
创建一个简单的集成模型，结合多个模型的预测（如平均）

In [ ]:
# 练习空间
# 在这里完成练习题


## 总结

本教程中，我们学习了：

1. **数据准备**：加载、探索、可视化 MOF 数据集
2. **特征工程**：特征缩放、选择
3. **模型训练**：
   - 线性回归（基线）
   - Ridge 回归（正则化）
   - 随机森林
   - XGBoost/LightGBM（梯度提升）
4. **超参数优化**：使用 GridSearchCV
5. **模型评估**：R²、RMSE、MAE、学习曲线
6. **可解释性分析**：
   - 特征重要性
   - SHAP 值分析
   - 单样本解释
7. **模型应用**：保存模型、预测新样本

### 关键发现：

- **最重要的特征**：比表面积、孔体积、孔径等几何特征对 CO₂ 吸附影响最大
- **最佳模型**：梯度提升模型（XGBoost/LightGBM）通常性能最好
- **可解释性**：SHAP 分析帮助我们理解模型决策，验证物理合理性

### 下一步：

- 第三天将学习**深度学习与图神经网络**
- 探索更复杂的结构表示方法
- 端到端学习 MOF 结构-性质关系

---

*教程结束*